# 05 · Crop recommender with SHAP

2,200 rows, 22 crops, 7 features. This is the module where explainability is easiest to show and most convincing in a viva: SHAP tells the farmer *why* a crop was recommended.


In [ ]:
# Kaggle setup — run this first in every notebook.
# Settings -> Accelerator: GPU T4 x2 | Internet: ON
import os, subprocess, sys
REPO = "/kaggle/working/SKIT-DS-2023-2027-015"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/shankerjoshi/SKIT-DS-2023-2027-015.git", REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxconverter-common", "shap"], check=True)
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


In [ ]:
from data.download import kaggle_download
from pathlib import Path
kaggle_download("atharvaingle/crop-recommendation-dataset", Path("data/raw/crop"))


In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, GroupKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score
import joblib

csv = next(Path("data/raw/crop").rglob("*.csv"))
df = pd.read_csv(csv)
X = df.drop(columns=["label"]); y = df["label"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
rf = RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1).fit(Xtr, ytr)
pred = rf.predict(Xte)
print("macro-F1:", round(f1_score(yte, pred, average="macro"), 4))
print("accuracy:", round((pred == yte).mean(), 4))


In [ ]:
import shap
explainer = shap.TreeExplainer(rf)
sv = explainer.shap_values(Xte.iloc[:200])
shap.summary_plot(sv, Xte.iloc[:200], show=True, max_display=10)


In [ ]:
# one farmer's recommendation, explained
row = Xte.iloc[[0]]
proba = rf.predict_proba(row)[0]
top = np.argsort(proba)[::-1][:3]
print("input:", row.to_dict("records")[0])
for i in top:
    print(f"  {rf.classes_[i]:12s} p={proba[i]:.3f}")
sv1 = explainer.shap_values(row)[0]
print("\nwhy:", {f: round(float(v), 3) for f, v in zip(X.columns, sv1)})


### Caveat to state in the report
The 2,200 rows are synthetic/augmented Indian averages, not Rajasthan farm records. The model learns plausible agronomy, not local truth — say that plainly and treat the output as advisory.
